# Debugging 调试支持教程 (v0.7)

本教程帮助你理解可观测性**三支柱联动**：日志 + 追踪 + 指标。

## 学习目标

1. 掌握性能指标收集：Counter、Gauge、Histogram
2. 学会使用 Timer 自动计时
3. 理解日志 + 追踪 + 指标的联动调试
4. 实践完整调试场景

In [ ]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "openai", "python-dotenv", "zhipuai", "-q"
])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

## 1. 可观测性三支柱

| 支柱 | 作用 | 本项目实现 |
|------|------|-----------|
| **Logging** | 记录发生了什么 | StructuredLogger |
| **Tracing** | 跟踪经过了哪里 | Tracer + Span |
| **Metrics** | 量化有多快 | MetricsCollector |

三者通过 `trace_id` 关联，形成完整的调试视角。

In [ ]:
# 导入所有可观测性组件
from rogue_rabbit.contracts.log import (
    LogLevel, LogEntry,
    MetricType, MetricPoint,
)
from rogue_rabbit.core.log_manager import (
    StructuredLogger,
    Tracer,
    MetricsCollector,
)
from rogue_rabbit.runtime.log_store import (
    InMemoryLogStore,
    InMemorySpanStore,
    InMemoryMetricStore,
)
import time

# 创建三个存储和管理器
log_store = InMemoryLogStore()
span_store = InMemorySpanStore()
metric_store = InMemoryMetricStore()

log = StructuredLogger(store=log_store, module="agent")
tracer = Tracer(store=span_store)
metrics = MetricsCollector(store=metric_store)

print("三支柱组件创建成功！")

## 2. 性能指标基础

三种指标类型：
- **Counter**: 单调递增计数（请求次数）
- **Gauge**: 可增可减的当前值（连接数）
- **Histogram**: 分布统计（响应时间）

In [ ]:
# Counter - 请求计数
metrics.increment("request.count")
metrics.increment("request.count")
metrics.increment("request.count", tags={"status": "200"})
metrics.increment("request.count", tags={"status": "500"})

# Gauge - 当前连接数
metrics.gauge("active_connections", value=5)
metrics.gauge("active_connections", value=3)

# Histogram - 响应时间
metrics.histogram("response.time", value=150.5, unit="ms")
metrics.histogram("response.time", value=200.3, unit="ms")
metrics.histogram("response.time", value=80.1, unit="ms")

# 查询和聚合
total = metric_store.aggregate("request.count", agg="sum")
avg_time = metric_store.aggregate("response.time", agg="avg")
max_time = metric_store.aggregate("response.time", agg="max")

print(f"请求总数: {total}")
print(f"平均响应时间: {avg_time:.1f}ms")
print(f"最大响应时间: {max_time:.1f}ms")

## 3. Timer 自动计时

使用 `timer()` 上下文管理器自动记录操作耗时。

In [ ]:
# 使用 timer 自动计时
with metrics.timer("db.query", tags={"table": "users"}):
    time.sleep(0.05)

with metrics.timer("db.query", tags={"table": "orders"}):
    time.sleep(0.03)

# Summary 一次性查看统计
summary = metrics.summary("db.query")
print("db.query 摘要:")
for key, val in summary.items():
    if val is not None:
        unit = "ms" if key != "count" else ""
        print(f"  {key}: {val:.1f}{unit}")

## 4. 完整调试场景

模拟一个 Agent 请求的完整流程，每步同时记录日志、追踪和指标。

In [ ]:
# 清空之前的数据
log_store.clear()
span_store.clear()
metric_store.clear()
tracer = Tracer(store=span_store)

# === 模拟 Agent 请求 ===
trace_id = tracer.start_trace("agent.chat", attributes={"user_id": "user1"})

# Step 1: 接收输入
log.info("收到用户输入", trace_id=trace_id, input="帮我写一个 hello world")
with metrics.timer("agent.input_processing"):
    time.sleep(0.01)

# Step 2: 权限检查
with tracer.start_span("permission.check", trace_id=trace_id) as span:
    log.info("检查权限", trace_id=trace_id, span_id=span.span_id, action="execute")
    time.sleep(0.02)
    span.add_event("checked", {"result": "allowed"})
    metrics.increment("permission.check.count", tags={"result": "allowed"})

# Step 3: LLM 调用
with tracer.start_span("llm.call", trace_id=trace_id) as llm_span:
    log.info("调用 LLM", trace_id=trace_id, span_id=llm_span.span_id, model="gpt-4")
    with metrics.timer("llm.call.duration", tags={"model": "gpt-4"}):
        time.sleep(0.06)
    llm_span.add_event("response_received", {"output_tokens": 80})
    metrics.increment("llm.call.count", tags={"model": "gpt-4"})
    metrics.histogram("llm.tokens.output", value=80, unit="tokens")

# Step 4: 保存记忆
with tracer.start_span("memory.save", trace_id=trace_id) as mem_span:
    log.info("保存对话记忆", trace_id=trace_id, span_id=mem_span.span_id)
    time.sleep(0.01)
    metrics.increment("memory.save.count")

log.info("请求处理完成", trace_id=trace_id)

print(f"模拟完成！trace_id={trace_id}")

## 5. 调试分析

使用 trace_id 关联查询三支柱数据，还原完整的请求上下文。

In [ ]:
print(f"{'=' * 50}")
print(f"调试分析 | trace_id={trace_id}")
print(f"{'=' * 50}")

# 1. 调用链分析
spans = tracer.get_trace(trace_id)
print(f"\n[调用链] {len(spans)} 个 Span:")
for s in spans:
    duration = f"{s.duration_ms:.1f}ms" if s.duration_ms else "N/A"
    indent = "  " if s.parent_span_id else ""
    print(f"  {indent}{s.name} | {duration} | {s.status.value}")

# 2. 日志上下文
logs = log_store.query()
print(f"\n[日志] {len(logs)} 条:")
for entry in logs:
    ctx_parts = [f"{k}={v}" for k, v in entry.context.items()]
    ctx_str = ", ".join(ctx_parts[:3])
    print(f"  [{entry.level.value:7s}] {entry.message} ({ctx_str})")

# 3. 性能指标
print(f"\n[指标]:")
for name in ["permission.check.count", "llm.call.count", "memory.save.count"]:
    total = metric_store.aggregate(name, agg="sum")
    print(f"  {name}: {int(total) if total else 0}")

llm_summary = metrics.summary("llm.call.duration")
if llm_summary["avg"] is not None:
    print(f"  llm.call.duration avg: {llm_summary['avg']:.1f}ms")

## 总结

### 可观测性三支柱

| 支柱 | 组件 | 作用 |
|------|------|------|
| Logging | `StructuredLogger` | 记录离散事件 |
| Tracing | `Tracer` | 跟踪请求路径 |
| Metrics | `MetricsCollector` | 量化性能 |

### 指标类型

| 类型 | 说明 | 示例 |
|------|------|------|
| Counter | 单调递增 | 请求次数 |
| Gauge | 当前值 | 连接数 |
| Histogram | 分布统计 | 响应时间 |

### 下一步

- 运行 `experiments/20_debugging.py`
- 查看 [roadmap](../docs/01-overview/roadmap.md) 了解后续版本